# ML Analysis of SMS Messages

This notebook presents an end-to-end machine learning workflow for classifying SMS messages as HAM (legitimate) or SPAM (unsolicited).

The approach follows a structured, hypothesis-driven process:

Exploratory Data Analysis (EDA) to understand class distribution, text characteristics, and linguistic patterns
Feature Engineering, combining TF-IDF text representations with additional structural features (e.g., message length, presence of URLs or digits)
Model Development, including baseline and comparative models such as Logistic Regression and Naive Bayes
Evaluation, with a focus on precision, recall, and F1-score, emphasizing recall for spam detection

The objective is to build an interpretable and effective classification model while demonstrating sound data science and machine learning practices.

---

**Date:** 2026-04-15

**Author:** Sergej Schweizer


## Table of Contents
- [Load data](#load-data)
- [Explorative Data Analysis](#explorative-data-analysis)
  - [Text messages comparison](#text-messages-comparison)
  - [Token Analysis](#token-analysis)
- [Feature Generation](#feature-generation)
  - [Base preprocessing](#base-preprocessing)
  - [Structural features](#structural-features)
  - [Semantic indicator features](#semantic-indicator-features)
  - [Phrase-level / contextual features](#phrase-level--contextual-features)
  - [Chained conditional feature](#chained-conditional-feature)
  - [TF-IDF most discriminative uni/bi/tri-grams features](#tf-idf-most-discriminative-unibitri-grams-features)
- [Modeling](#modeling)
  - [Base splitting and format to matrix format](#base-splitting-and-format-to-matrix-format)
  - [Cross-Validation Setup](#cross-validation-setup)
  - [Best Model: Confusion Matrix](#best-model-confusion-matrix)
  - [Best Model: Feature Importance](#best-model-feature-importance)
  - [Best Model: False Negatives](#best-model-false-negatives)
  - [Best Model: False Positives](#best-model-false-positives)


In [1]:
import warnings
warnings.filterwarnings("ignore")

# Import dependencies and configure project paths.
from pathlib import Path
import sys
 
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
 
from src.modeling import (
     LogisticRegressionStrategy,
     MultinomialNBStrategy,
     RandomForestStrategy,
)
from src.training_utils import run_strategy_pipeline
from src.utils import (
     add_selected_tfidf_features,
     preprocess,
     top_ngram_summary,
)
 
optuna.logging.set_verbosity(optuna.logging.WARNING)

### Load data

In [2]:
# Load the SMS dataset and standardize the base dataframe.
df = pd.read_csv(
    "../data/raw/spam.csv", 
    encoding="cp1252", 
    usecols=["v1", "v2"]
    ).rename(columns={"v1": "label", "v2": "message"})
df.sample(10)

,label,message
2155,ham,What year. And how many miles.
3825,ham,"Goodmorning,my grandfather expired..so am on l..."
2559,ham,I also thk too fast... Xy suggest one not me. ...
1990,ham,HI DARLIN IVE JUST GOT BACK AND I HAD A REALLY...
3114,ham,I wait 4 Ì_ inside da car park...
2086,ham,Dude how do you like the buff wind.
2325,ham,Apps class varaya elaya.
4507,ham,Not able to do anything.
2564,ham,"Under the sea, there lays a rock. In the rock,..."
220,ham,Ok no prob. Take ur time.


## Explorative Data Analysis
Class imbalance immediately suggests we should focus on recall for spam.

In [3]:
# Inspect class balance between ham and spam messages.
df["label"].value_counts(normalize=True)

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

### Text messages comparison
Assumption: spam message are at average longer than ham messages

In [4]:
# Create basic length features and compare message characteristics by class.
df["length"] = df["message"].apply(len).astype(int)
df.groupby("label")["length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4825.0,71.023627,58.016023,2.0,33.0,52.0,92.0,910.0
spam,747.0,138.866131,29.183082,13.0,132.5,149.0,157.0,224.0


**New Feature**: length could serve as good indicator

### Token Analysis
**Unique uni-grams for all classes**

In [5]:
# Compute token frequency statistics for exploratory text analysis.
freq_df = (
    df["message"]
    .apply(preprocess)   
    .explode()           
    .value_counts()
    .rename_axis("word")
    .reset_index(name="count")
)

freq_df["rel_freq"] = freq_df["count"] / freq_df["count"].sum().astype(int)

freq_df.head(20)

,word,count,rel_freq
0,your,677,0.011067
1,call,576,0.009416
2,2,478,0.007814
3,now,478,0.007814
4,im,471,0.007699
5,not,430,0.007029
6,can,405,0.006620
7,get,386,0.006310
8,ur,384,0.006277
9,will,378,0.006179


**Insights**
- Global unigram frequencies provide limited predictive power, as they are dominated by common and non-discriminative words; meaningful signals emerge only when comparing class-specific distributions (spam vs. ham).

**Most frequent uni-grams per class**

In [6]:
# Summarize top unigram differences between spam and ham.
result_uni = top_ngram_summary(df, n=1, top_k=25)
result_uni

,spam_ngram,spam_count,spam_rel_count,ham_ngram,ham_count,ham_rel_count
0,call,347,0.024629,im,458,0.009727
1,your,263,0.018667,your,414,0.008792
2,free,216,0.015331,not,406,0.008623
3,now,189,0.013415,can,376,0.007985
4,2,173,0.012279,will,334,0.007093
5,txt,150,0.010647,2,305,0.006478
6,ur,144,0.010221,get,303,0.006435
7,from,128,0.009085,its,302,0.006414
8,mobile,123,0.00873,just,289,0.006138
9,text,120,0.008517,now,289,0.006138


**Insights**
- Clear lexical separation: Spam messages use action-driven, transactional language (call, free, claim, reply), while ham messages are conversational and informal (im, ok, just, will).
- Call-to-action pattern: Spam strongly relies on urgency and commands (now, call, claim), indicating conversion-focused intent.
- Channel-specific vocabulary: Tokens like txt, mobile, reply, stop appear almost exclusively in spam, reflecting automated messaging and subscription flows.
- Numeric tokens as signals: Numbers (2, 4) are more frequent in spam, often encoding offers, shortcodes, or instructions.
- Shared words differ in intensity: Words like your and now appear in both classes but are significantly more frequent in spam, indicating stronger targeting and urgency.
- Language style contrast: Ham messages contain more natural, human-like expressions (im, ok), while spam shows templated, structured wording (claim, prize).
- Higher concentration in spam: Spam vocabulary is more repetitive and dominated by fewer tokens, making it easier to model statistically.
- Semantic structure in spam: Spam tokens cluster into categories (actions, incentives, channels), showing predictable, non-random patterns.
- Strong separability: The clear differences in token distributions indicate that spam vs. ham is highly separable in feature space.

**New features**
- TF-IDF features: Weight tokens by importance across the corpus to downweight common words and highlight discriminative spam indicators (free, call, prize).
- Log-odds / frequency ratio features: Use class-specific token ratios to explicitly capture how strongly a word indicates spam vs. ham.
- N-gram features: Include bigrams/trigrams (e.g. â€œcall nowâ€, â€œfree entryâ€) to capture spam-specific phrases and context.
- Semantic indicator features: Binary flags for categories like call-to-action, incentives, and channel terms (e.g. contains â€œtxtâ€, â€œreplyâ€, â€œstopâ€).
- Structural features: Message-level signals such as token count, digit count, and proportion of numeric or uppercase characters.



**Same bi and trigrams in both classes**

In [7]:
# Compare discriminative bi-grams and tri-grams across classes.
bigram_summary = top_ngram_summary(df, n=2, top_k=None).assign(n=2)
trigram_summary = top_ngram_summary(df, n=3, top_k=None).assign(n=3)

combined = pd.concat([bigram_summary, trigram_summary], ignore_index=True)

spam_df = (
    combined[["n", "spam_ngram", "spam_count", "spam_rel_count"]]
    .dropna()
    .rename(columns={"spam_ngram": "phrase"})
)

ham_df = (
    combined[["n", "ham_ngram", "ham_count", "ham_rel_count"]]
    .dropna()
    .rename(columns={"ham_ngram": "phrase"})
)

exact_matches = (
    spam_df.merge(ham_df, on=["n", "phrase"], how="inner")
           .sort_values(["spam_rel_count", "ham_rel_count"], ascending=False)
           .reset_index(drop=True)
)

exact_matches.head(25)

,n,phrase,spam_count,spam_rel_count,ham_count,ham_rel_count
0,2,please call,44,0.003298,9,0.000213
1,2,po box,24,0.001799,1,0.000024
2,2,call now,20,0.001499,6,0.000142
3,2,from landline,20,0.001499,2,0.000047
4,2,every week,19,0.001424,2,0.000047
5,2,find out,18,0.001349,12,0.000284
6,2,out who,18,0.001349,1,0.000024
7,2,customer service,17,0.001274,2,0.000047
8,2,had your,15,0.001124,1,0.000024
9,2,just call,15,0.001124,1,0.000024


**Insights**
- Shared structure, different intent:
Phrases like â€œcall nowâ€, â€œplease callâ€, â€œcall backâ€ appear in both classes, but occur much more frequently in spam, indicating the same linguistic template used with different intent (transactional vs. social).
- Action verbs dominate cross-class overlap:
Most shared n-grams revolve around verbs like call, text, send, get, confirming that actions are central in both, but spam amplifies them.
- Spam intensifies common phrases:
Even when phrases exist in ham (â€œcall nowâ€, â€œyour phoneâ€), their relative frequency is significantly higher in spam, showing stronger urgency and targeting.
- Spam-specific context around shared phrases:
Shared n-grams in spam are often embedded in promotional or system-like contexts (â€œcustomer serviceâ€, â€œfree callâ€, â€œmobile numberâ€), while ham usage remains situational.
- Ham-specific conversational phrases stand out:
Phrases like â€œdont knowâ€, â€œhas beenâ€ appear much more in ham, reflecting natural dialogue and uncertainty, which is largely absent in spam.
- Numeric and shortcode patterns persist in shared space:
Even overlapping phrases like â€œ4 urâ€, â€œ2 stopâ€ show that numeric shorthand is more structured and frequent in spam, hinting at templated messaging.

**New Features**
- Weighted n-gram features (TF-IDF on n-grams):
Capture importance of phrases like â€œcall nowâ€, â€œfree callâ€, which are much more indicative than single tokens.
- Log-odds on n-grams:
Compute class-specific ratios for phrases (e.g. â€œplease callâ€), which provide stronger discrimination than word-level log-odds.
- Call-to-action phrase indicators:
Binary features for patterns like (call/text/send + object/time) â†’ e.g. â€œcall nowâ€, â€œtext backâ€.
Phrase intensity features:
Measure how much more frequently a phrase appears in spam vs. ham (e.g. spam_rel / ham_rel), capturing urgency amplification.
- Contextual phrase grouping:
Group n-grams into categories (communication, promotion, interaction) and use counts per group as features.

## Feature Generation

### Base preprocessing

In [8]:
# Apply base text preprocessing and clean core columns.
df_base = (
    df.assign(
        message=lambda x: x["message"].fillna("").astype(str),
        message_lower=lambda x: x["message"].str.lower(),
    )
)

display(df_base[["label", "message_lower"]].sample(3, random_state=42))

,label,message_lower
3245,ham,"funny fact nobody teaches volcanoes 2 erupt, t..."
944,ham,i sent my scores to sophas and i had to do sec...
1044,spam,we know someone who you know that fancies you....


### Structural features

In [9]:
# Generate structural message features (counts, ratios, and symbol patterns).
currency_pattern = r"\u00A3|\$|\u20AC"

df_structural = (
    df_base.assign(
        char_count=lambda x: x["message"].str.len().astype(int),
        word_count=lambda x: x["message"].str.split().str.len().astype(int),
        digit_count=lambda x: x["message"].str.count(r"\d").astype(int),
        uppercase_count=lambda x: x["message"].str.count(r"[A-Z]").astype(int),
        exclamation_count=lambda x: x["message"].str.count(r"!").astype(int),
        question_count=lambda x: x["message"].str.count(r"\?").astype(int),
        currency_count=lambda x: x["message"].str.count(currency_pattern).astype(int),
    )
)

df_structural = df_structural.assign(
    numeric_ratio=lambda x: (x["digit_count"] / x["char_count"].replace(0, np.nan)).fillna(0.0),
    uppercase_ratio=lambda x: (x["uppercase_count"] / x["char_count"].replace(0, np.nan)).fillna(0.0),
)

structural_feature_cols = [
    "message_lower",
    "char_count",
    "word_count",
    "digit_count",
    "uppercase_count",
    "exclamation_count",
    "question_count",
    "currency_count",
    "numeric_ratio",
    "uppercase_ratio",
]

display(df_structural[structural_feature_cols].sample(3, random_state=42))

,message_lower,char_count,word_count,digit_count,uppercase_count,exclamation_count,question_count,currency_count,numeric_ratio,uppercase_ratio
3245,"funny fact nobody teaches volcanoes 2 erupt, t...",151,26,5,3,0,0,0,0.033113,0.019868
944,i sent my scores to sophas and i had to do sec...,221,43,0,3,0,0,0,0.000000,0.013575
1044,we know someone who you know that fancies you....,101,19,17,9,0,0,0,0.168317,0.089109


### Semantic indicator features

In [10]:
# Generate structural message features (counts, ratios, and symbol patterns).
df_semantic = (
    df_structural.assign(
        has_call_to_action=lambda x: x["message_lower"].str.contains(
            r"\b(?:call|txt|text|reply|send|claim|stop|win|urgent|contact)\b"
        ).astype(int),
        has_incentive=lambda x: x["message_lower"].str.contains(
            r"\b(?:free|prize|cash|award|won|winner|bonus|gift)\b"
        ).astype(int),
        has_channel_term=lambda x: x["message_lower"].str.contains(
            r"\b(?:txt|text|mobile|phone|call|reply|stop)\b"
        ).astype(int),
        has_time_pressure=lambda x: x["message_lower"].str.contains(
            r"\b(?:now|today|only|limited|urgent|immediately|expires?)\b"
        ).astype(int),
        has_money=lambda x: x["message_lower"].str.contains(
            rf"\b\d+\s?(?:p|ppm|min|gbp|usd|eur)\b|{currency_pattern}"
        ).astype(int),
    )
)

semantic_feature_cols = [
    "message_lower",
    "has_call_to_action",
    "has_incentive",
    "has_channel_term",
    "has_time_pressure",
    "has_money",
]

display(df_semantic[semantic_feature_cols].sample(3, random_state=42))

,message_lower,has_call_to_action,has_incentive,has_channel_term,has_time_pressure,has_money
3245,"funny fact nobody teaches volcanoes 2 erupt, t...",0,0,0,0,0
944,i sent my scores to sophas and i had to do sec...,1,0,0,0,0
1044,we know someone who you know that fancies you....,1,0,1,0,1


### Phrase-level / contextual features

In [11]:
# Build phrase-level contextual features from key spam-related expressions.
df_features = (
    df_semantic.assign(
        has_call_now=lambda x: x["message_lower"].str.contains(r"\bcall now\b").astype(int),
        has_text_back=lambda x: x["message_lower"].str.contains(
            r"\b(?:text back|txt back|reply now)\b"
        ).astype(int),
        has_free_entry=lambda x: x["message_lower"].str.contains(r"\bfree entry\b").astype(int),
        communication_terms=lambda x: x["message_lower"].str.count(
            r"\b(?:call|txt|text|reply|send|contact|mobile|phone)\b"
        ).astype(int),
        promotion_terms=lambda x: x["message_lower"].str.count(
            r"\b(?:free|prize|cash|bonus|offer|award|gift|win|won)\b"
        ).astype(int),
        interaction_terms=lambda x: x["message_lower"].str.count(
            r"\b(?:chat|talk|meet|join|subscribe|claim|stop)\b"
        ).astype(int),
    )
)

phrase_feature_cols = [
    "message_lower",
    "has_call_now",
    "has_text_back",
    "has_free_entry",
    "communication_terms",
    "promotion_terms",
    "interaction_terms",
]

display(df_features[phrase_feature_cols].sample(3, random_state=42))

,message_lower,has_call_now,has_text_back,has_free_entry,communication_terms,promotion_terms,interaction_terms
3245,"funny fact nobody teaches volcanoes 2 erupt, t...",0,0,0,0,0,0
944,i sent my scores to sophas and i had to do sec...,0,0,0,1,0,0
1044,we know someone who you know that fancies you....,0,0,0,1,0,0


### Chained conditional feature

In [12]:
# Create chained conditional interaction features to strengthen spam signals.
df_features = (
    df_features.assign(
        spam_signal_strength=lambda x: np.select(
            [
                (x["has_call_to_action"] == 1) & (x["has_incentive"] == 1) & (x["has_time_pressure"] == 1),
                (x["has_call_to_action"] == 1) & (x["has_incentive"] == 1),
                (x["has_call_to_action"] == 1) | (x["has_incentive"] == 1),
            ],
            [3, 2, 1],
            default=0,
        ).astype(int)
    )
)

conditional_feature_cols = [
    "label",
    "message_lower",
    "spam_signal_strength",
]

display(df_features[conditional_feature_cols].sample(3, random_state=42))

,label,message_lower,spam_signal_strength
3245,ham,"funny fact nobody teaches volcanoes 2 erupt, t...",0
944,ham,i sent my scores to sophas and i had to do sec...,1
1044,spam,we know someone who you know that fancies you....,1


### TF-IDF most discriminative uni/bi/tri-grams features

This step transforms raw SMS text into a compact and informative feature space. First, TF-IDF is used to convert messages into numerical representations based on word importance. Then, chi2 feature selection retains only the most discriminative terms for spam classification, reducing dimensionality. Finally, these sparse text features are combined with handcrafted dense features (e.g., message length, call-to-action indicators) to create a unified feature matrix for model training.

In [13]:
# Build TF-IDF features and preview selected high-signal terms.
df_features = df_features.copy()
df_features = df_features.fillna(0)

dense_exclude_cols = {"label", "message", "message_lower"}
dense_feature_cols = [
    col
    for col in df_features.select_dtypes(include=["number", "bool"]).columns
    if col not in dense_exclude_cols and not col.startswith("tfidf__")
]



y_preview = df_features["label"].eq("spam").astype(int)

df_tfidf_preview, selected_tfidf_cols_preview, tfidf_preview, selector_preview = add_selected_tfidf_features(
    df=df_features.copy(),
    text_col="message_lower",
    target=y_preview,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.90,
    max_features=3000,
    k=300,
    prefix="tfidf__",
)

df_tfidf_preview[selected_tfidf_cols_preview] = df_tfidf_preview[selected_tfidf_cols_preview].fillna(0).astype(float)

print(f"dense features: {len(dense_feature_cols)}")
print(f"selected tfidf features: {len(selected_tfidf_cols_preview)}")

display(df_tfidf_preview[selected_tfidf_cols_preview].sample(3, random_state=42))

dense features: 22
selected tfidf features: 300


,tfidf__000,tfidf__03,tfidf__04,tfidf__0800,tfidf__08000839402,tfidf__08000839402 or,tfidf__08000930705,tfidf__100,tfidf__1000,tfidf__1000 cash,...,tfidf__www,tfidf__you are awarded,tfidf__you have,tfidf__you have new,tfidf__you have won,tfidf__your,tfidf__your 2003,tfidf__your 2003 account,tfidf__your free,tfidf__your mobile
3245,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
944,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1044,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Modeling

### Base splitting and format to matrix format

In [14]:
# Create train/test split and assemble final dense plus TF-IDF feature matrices.
TARGET_COL = "label"
TEXT_COL = "message_lower"
TFIDF_PREFIX = "tfidf__"
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Use preview dataframe (already containing selected TF-IDF features).
df_model_input = df_tfidf_preview.copy()
dense_exclude_cols = {TARGET_COL, "message", TEXT_COL}

y_all = df_model_input[TARGET_COL].eq("spam").astype(int)

train_idx, test_idx = train_test_split(
    df_model_input.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_all,
)

df_train = df_model_input.loc[train_idx].copy()
df_test = df_model_input.loc[test_idx].copy()

y_train = y_all.loc[train_idx]
y_test = y_all.loc[test_idx]

dense_feature_cols = [
    col
    for col in df_model_input.select_dtypes(include=["number", "bool"]).columns
    if col not in dense_exclude_cols and not col.startswith(TFIDF_PREFIX)
]

selected_tfidf_cols = [
    col for col in df_model_input.columns if col.startswith(TFIDF_PREFIX)
]

df_train[dense_feature_cols] = df_train[dense_feature_cols].fillna(0).astype(float)
df_test[dense_feature_cols] = df_test[dense_feature_cols].fillna(0).astype(float)

final_feature_cols = dense_feature_cols + selected_tfidf_cols
X_train = df_train[final_feature_cols].fillna(0).astype(float)
X_test_final = df_test[final_feature_cols].fillna(0).astype(float)

print(f"train rows: {len(X_train)} | test rows: {len(X_test_final)}")
print(f"dense features: {len(dense_feature_cols)}")
print(f"selected tfidf features: {len(selected_tfidf_cols)}")
print(f"total model features: {len(final_feature_cols)}")


train rows: 4457 | test rows: 1115
dense features: 22
selected tfidf features: 300
total model features: 322


### K-Fold with Bayes Optimization

For each model, Bayesian optimization is used to search the hyperparameter space efficiently instead of testing parameter combinations manually or through exhaustive grid search. In every optimization trial, a candidate parameter set is evaluated with k-fold cross-validation, and the mean recall score is used as the objective value. This allows the workflow to identify the best-performing configuration for each model while keeping the tuning process computationally efficient. After optimization, the best version of each model is trained on the full training set and then evaluated on the holdout test set.

In [15]:
# Train and tune all modeling strategies with Optuna.
strategies = [
    LogisticRegressionStrategy(),
    RandomForestStrategy(),
    MultinomialNBStrategy()
]

all_results = {}

for strategy in strategies:
    result = run_strategy_pipeline(
        strategy=strategy,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test_final,
        y_test=y_test,
        df_test=df_test,
        text_col=TEXT_COL,
        feature_names=final_feature_cols,
        n_trials=30,
        top_n=20,
    )
    all_results[strategy.name()] = result

Optimizing logistic_regression:   0%|          | 0/30 [00:00<?, ?trial/s]

Optimizing random_forest:   0%|          | 0/30 [00:00<?, ?trial/s]

Optimizing multinomial_nb:   0%|          | 0/30 [00:00<?, ?trial/s]

In [16]:
# Aggregate and rank model metrics for strategy comparison.
comparison_df = pd.DataFrame([
    {
        "model": model_name,
        "accuracy": result["metrics"]["accuracy"],
        "precision": result["metrics"]["precision"],
        "recall": result["metrics"]["recall"],
        "f1": result["metrics"]["f1"],
        "roc_auc": result["metrics"].get("roc_auc", np.nan),
        "best_params": result["best_params"],
    }
    for model_name, result in all_results.items()
]).sort_values(["recall", "f1", "roc_auc"], ascending=False)

comparison_df

,model,accuracy,precision,recall,f1,roc_auc,best_params
1,random_forest,0.982063,0.938776,0.926174,0.932432,0.986445,"{'n_estimators': 107, 'max_depth': 8, 'min_sam..."
0,logistic_regression,0.976682,0.901961,0.926174,0.913907,0.992490,{'C': 2.6772933611447796}
2,multinomial_nb,0.922870,0.647887,0.926174,0.762431,0.949682,"{'alpha': 0.0035434848393501384, 'fit_prior': ..."


### Best Model: Confusion Matrix

In [17]:
# Select the top model and print detailed evaluation metrics.
best_model_name = comparison_df.iloc[0]["model"]
best_result = all_results[best_model_name]
print("Best model:", best_model_name)
print(best_result["metrics"]["classification_report"])

Best model: random_forest
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.94      0.93      0.93       149

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



### Best Model: Feature Importance

In [18]:
# Display feature-importance view for the best-performing model.
best_result["importance"]

,feature,importance
0,numeric_ratio,0.122014
1,digit_count,0.102416
2,spam_signal_strength,0.083677
3,communication_terms,0.076596
4,uppercase_count,0.060489
5,length,0.053816
6,char_count,0.052540
7,has_call_to_action,0.050241
8,has_channel_term,0.048300
9,word_count,0.039980


### Best Model: False Negatives

In [19]:
# Review false negatives from the best model.
best_result["false_negatives"].head(10)

,message_lower,y_true,y_pred,pred_score
0,for sale - arsenal dartboard. good condition b...,1,0,0.029988
1,"latest news! police station toilet stolen, cop...",1,0,0.038180
2,"do you realize that in about 40 years, we'll h...",1,0,0.097537
3,hi ya babe x u 4goten bout me?' scammers getti...,1,0,0.109628
4,bought one ringtone and now getting texts cost...,1,0,0.122175
5,romcapspam everyone around should be respondin...,1,0,0.136017
6,adult 18 content your video will be with you s...,1,0,0.243384
7,rct' thnq adrian for u text. rgds vatian,1,0,0.258437
8,freemsg>fav xmas tones!reply real,1,0,0.370525
9,ringtoneking 84484,1,0,0.394761


### Best Model: False Positives

In [20]:
# Review false positives from the best model.
best_result["false_positives"].head(10)

,message_lower,y_true,y_pred,pred_score
0,your opinion about me? 1. over 2. jada 3. kusr...,0,1,0.835874
1,"hi jon, pete here, ive bin 2 spain recently & ...",0,1,0.738430
2,this is ur face test ( 1 2 3 4 5 6 7 8 9 &lt;...,0,1,0.660678
3,hi this is yijue... it's regarding the 3230 te...,0,1,0.607522
4,hey pple...$700 or $900 for 5 nights...excelle...,0,1,0.572282
5,solve d case : a man was found murdered on &l...,0,1,0.569629
6,my no. in luton 0125698789 ring me if ur aroun...,0,1,0.552461
7,\hey! do u fancy meetin me at 4 at cha åð hav ...,0,1,0.542133
8,sorry im stil fucked after last nite went tobe...,0,1,0.541524
